# Acceptance Rate Analysis — Own Re-collected Data (Same Months as AIDev)

This notebook computes acceptance/rejection rates for AI-agent fix PRs using **our own re-collected data**
(`mabujadallah/GitHub-Agentic-PR-Dataset`), filtered to the **same date range as the AIDev dataset**
(Dec 2024 – Jul 2025).

This allows direct comparison with the AIDev acceptance-rate notebook.

**Agents studied:** Copilot, Devin, Cursor, Claude Code (same 4 as the paper).

**Produces:**
- `results/own_data_report_figures/` — PNG figures
- `results/own_data_report.txt` — text summary

In [ ]:
%pip install matplotlib seaborn scipy pyarrow fsspec requests

## Imports & Setup

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats as sp_stats

%matplotlib inline

## Paths

In [ ]:
OUT_DIR = Path("results")
FIG_DIR = OUT_DIR / "own_data_report_figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = OUT_DIR / "own_data_report.txt"

## Load Own Re-collected Data & AIDev Date Range

In [ ]:
HF_BASE = "https://huggingface.co/datasets/mabujadallah/GitHub-Agentic-PR-Dataset/resolve/main"
AIDEV_BASE = "hf://datasets/hao-li/AIDev"

AGENTS_STUDIED = ["Copilot", "Devin", "Cursor", "Claude_Code"]

# ── Step 1: Determine the exact AIDev date range ──
print("Loading AIDev PRs to determine date range ...")
aidev_prs = pd.read_parquet(
    f"{AIDEV_BASE}/all_pull_request.parquet",
    columns=["created_at", "agent"],
    filters=[("agent", "in", AGENTS_STUDIED)],
)
aidev_dates = pd.to_datetime(aidev_prs["created_at"], utc=True)
AIDEV_MIN = aidev_dates.min()
AIDEV_MAX = aidev_dates.max()
print(f"  AIDev date range: {AIDEV_MIN} to {AIDEV_MAX}")
print(f"  AIDev PRs (4 agents): {len(aidev_prs):,}")
del aidev_prs, aidev_dates  # free memory

# ── Step 2: Load our own data ──
print("\nLoading own re-collected data from Hugging Face ...")
all_prs_raw = pd.read_parquet(f"{HF_BASE}/fix_prs_only.parquet")
print(f"  Total fix PRs loaded: {len(all_prs_raw):,}")
print(f"  Date range (full): {all_prs_raw['created_at'].min()} to {all_prs_raw['created_at'].max()}")

# ── Step 3: Keep only agent PRs (4 agents) ──
all_prs_raw = all_prs_raw[all_prs_raw["source"] == "agent"].copy()
all_prs_raw = all_prs_raw[all_prs_raw["agent"].isin(AGENTS_STUDIED)].copy()
print(f"  Agent fix PRs (4 agents): {len(all_prs_raw):,}")

# ── Step 4: Filter to AIDev date range ──
all_prs_raw["created_dt"] = pd.to_datetime(all_prs_raw["created_at"], utc=True)
all_prs_raw = all_prs_raw[
    (all_prs_raw["created_dt"] >= AIDEV_MIN) &
    (all_prs_raw["created_dt"] <= AIDEV_MAX)
].copy()
print(f"  Agent fix PRs within AIDev date range: {len(all_prs_raw):,}")

print(f"\nAgent distribution (within AIDev months):")
for agent, cnt in all_prs_raw["agent"].value_counts().items():
    print(f"    {agent:20s} {cnt:>8,}")
print(f"  State distribution:")
print(f"    closed: {(all_prs_raw['state'] == 'closed').sum():,}")
print(f"    open:   {(all_prs_raw['state'] == 'open').sum():,}")

## Filter: Closed Fix PRs Only

In [ ]:
fix_prs = all_prs_raw[all_prs_raw["state"] == "closed"].copy()
fix_prs["is_merged"] = fix_prs["merged_at"].notna()

print(f"Closed agent fix PRs (4 agents, AIDev months): {len(fix_prs):,}")
print(f"  Merged:   {fix_prs['is_merged'].sum():,}")
print(f"  Rejected: {(~fix_prs['is_merged']).sum():,}")
print(f"  Repos:    {fix_prs['repo_name'].nunique():,}")

In [ ]:
display(fix_prs.sample(10))

## Helper Functions

In [ ]:
report_lines: list[str] = []


def section(title: str):
    report_lines.append("")
    report_lines.append("=" * 70)
    report_lines.append(f"  {title}")
    report_lines.append("=" * 70)
    print(f"\n{'=' * 70}\n  {title}\n{'=' * 70}")


def out(line: str = ""):
    report_lines.append(line)
    print(line)


def merge_rate(df: pd.DataFrame) -> tuple[int, int, float]:
    merged = int(df["merged_at"].notna().sum())
    total = len(df)
    return merged, total, merged / total * 100 if total else 0

## Dataset Overview

In [ ]:
section("Dataset Overview (Own Data — Closed Fix PRs, AIDev Months)")
out(f"Data source        : mabujadallah/GitHub-Agentic-PR-Dataset (re-collected)")
out(f"AIDev date filter  : {AIDEV_MIN} to {AIDEV_MAX}")
out(f"Agents studied     : {', '.join(AGENTS_STUDIED)}")
out(f"Total closed fix PRs : {len(fix_prs):,}")
out(f"Repositories       : {fix_prs['repo_name'].nunique():,}")

out("\nPer-agent fix PR counts:")
for agent, cnt in fix_prs["agent"].value_counts().items():
    out(f"  {agent:20s} {cnt:>8,}")

## Acceptance / Rejection Rates

**Core metric:** A closed PR is *accepted* if `merged_at` is not null, otherwise *rejected*.

In [ ]:
section("Acceptance / Rejection Rates")

# Overall
total_merged, total_count, total_rate = merge_rate(fix_prs)
total_rejected = total_count - total_merged
rejection_rate = 100 - total_rate

out(f"Overall (4 agents combined):")
out(f"  Accepted (merged)      : {total_merged:,} / {total_count:,} = {total_rate:.2f}%")
out(f"  Rejected (closed only) : {total_rejected:,} / {total_count:,} = {rejection_rate:.2f}%")

# Per agent
out(f"\nPer-agent breakdown:")
agent_stats = []
for agent_name in sorted(fix_prs["agent"].unique()):
    grp = fix_prs[fix_prs["agent"] == agent_name]
    m, t, r = merge_rate(grp)
    rej = t - m
    rej_r = 100 - r
    agent_stats.append({
        "Agent": agent_name,
        "#Accepted": m,
        "#Rejected": rej,
        "#Total": t,
        "%Acceptance": r,
        "%Rejection": rej_r,
    })
    out(f"  {agent_name:20s}  {m:>6,} / {t:>6,} merged = {r:.1f}% acceptance  ({rej:,} rejected = {rej_r:.1f}%)")

agent_stats_df = pd.DataFrame(agent_stats)
print("\n")
display(agent_stats_df)

In [ ]:
# ── Acceptance & rejection rate bar chart ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overall acceptance vs rejection
bars = axes[0].bar(["Accepted", "Rejected"], [total_rate, rejection_rate],
                   color=["#2ca02c", "#d62728"], width=0.5, edgecolor="white")
for bar, val in zip(bars, [total_rate, rejection_rate]):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                 f"{val:.1f}%", ha="center", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Rate (%)")
axes[0].set_title("Overall Fix PR Outcome (Own Data, AIDev Months)")
axes[0].set_ylim(0, 110)

# Right: per-agent acceptance rate
agents_sorted = agent_stats_df.sort_values("%Acceptance", ascending=True)
colors = sns.color_palette("tab10", len(agents_sorted))
bars2 = axes[1].barh(agents_sorted["Agent"], agents_sorted["%Acceptance"], color=colors)
for bar, val in zip(bars2, agents_sorted["%Acceptance"]):
    axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                 f"{val:.1f}%", va="center", fontsize=10, fontweight="bold")
axes[1].set_xlabel("Acceptance Rate (%)")
axes[1].set_title("Fix PR Acceptance Rate by Agent (Own Data)")
axes[1].set_xlim(0, 110)

fig.suptitle("Own Data (AIDev Months) — Fix PR Acceptance / Rejection", fontsize=14)
fig.tight_layout()
fig.savefig(FIG_DIR / "acceptance_rejection_rate.png", dpi=150, bbox_inches="tight")
plt.show()
out(f"\n  -> Figure saved: {FIG_DIR / 'acceptance_rejection_rate.png'}")

## Table: Acceptance and Merge Time Statistics

Same format as the AIDev notebook (replicates Table 2 from the paper).

In [ ]:
section("Table: Acceptance and Merge Time Statistics of Fix PRs")


def compute_row(label, df):
    accepted = int(df["merged_at"].notna().sum())
    rejected = int(df["merged_at"].isna().sum())
    total = accepted + rejected
    pct = accepted / total * 100 if total else 0
    merged = df[df["merged_at"].notna()].copy()
    if len(merged):
        merged["hours"] = (
            pd.to_datetime(merged["merged_at"], utc=True)
            - pd.to_datetime(merged["created_at"], utc=True)
        ).dt.total_seconds() / 3600
        med_hours = merged["hours"].median()
    else:
        med_hours = float("nan")
    return [label, f"{accepted:,}", f"{rejected:,}", f"{total:,}", f"{pct:.1f}%", f"{med_hours:.2f}h"]


rows = [compute_row("All 4 Agents", fix_prs)]
for agent_name in sorted(fix_prs["agent"].unique()):
    rows.append(compute_row(f"  {agent_name}", fix_prs[fix_prs["agent"] == agent_name]))

summary_df = pd.DataFrame(rows, columns=["Category", "#Accepted", "#Rejected", "#Total", "%Acceptance", "Median Time to Merge"])
print("Acceptance and merge time statistics of fix PRs (own data, AIDev months).\n")
display(summary_df)

# Export as text
out(summary_df.to_string(index=False))

## Time to Merge Distribution

In [ ]:
section("Time to Merge")

merged_prs = fix_prs[fix_prs["merged_at"].notna()].copy()
merged_prs["created_dt"] = pd.to_datetime(merged_prs["created_at"], utc=True)
merged_prs["merged_dt"] = pd.to_datetime(merged_prs["merged_at"], utc=True)
merged_prs["hours_to_merge"] = (merged_prs["merged_dt"] - merged_prs["created_dt"]).dt.total_seconds() / 3600

out(f"Merged fix PRs: {len(merged_prs):,}")
out(f"Overall median time to merge: {merged_prs['hours_to_merge'].median():.2f} hours")

out("\nPer-agent median time to merge:")
for agent_name in sorted(merged_prs["agent"].unique()):
    grp = merged_prs[merged_prs["agent"] == agent_name]
    med = grp["hours_to_merge"].median()
    out(f"  {agent_name:20s}  {med:.2f} hours  (n={len(grp):,})")

In [ ]:
# Time-to-merge box plot per agent
agents_ordered = sorted(merged_prs["agent"].unique())
data_ttm = []
labels_ttm = []
for agent_name in agents_ordered:
    grp = merged_prs[merged_prs["agent"] == agent_name]
    # Clip at 95th percentile for visualization
    vals = grp["hours_to_merge"].clip(upper=grp["hours_to_merge"].quantile(0.95)).values
    data_ttm.append(vals)
    labels_ttm.append(agent_name)

fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot(data_ttm, labels=labels_ttm, patch_artist=True,
                widths=0.5, showfliers=False,
                medianprops=dict(color="black", linewidth=2))
palette = sns.color_palette("tab10", len(agents_ordered))
for patch, c in zip(bp["boxes"], palette):
    patch.set_facecolor(c)
for i, vals in enumerate(data_ttm):
    med = np.median(vals)
    ax.text(i + 1, med * 1.15 + 0.5, f"{med:.1f}h", ha="center", fontsize=10, fontweight="bold")
ax.set_ylabel("Hours to Merge")
ax.set_title("Time to Merge — Own Data Fix PRs by Agent (AIDev Months)")
fig.tight_layout()
fig.savefig(FIG_DIR / "time_to_merge_by_agent.png", dpi=150)
plt.show()
out(f"  -> Figure saved: {FIG_DIR / 'time_to_merge_by_agent.png'}")

## Monthly Acceptance Rate Trends

In [ ]:
section("Monthly Acceptance Rate")

fix_prs["created_month"] = pd.to_datetime(fix_prs["created_at"], utc=True).dt.to_period("M")

# Overall monthly
monthly_all = (
    fix_prs.groupby("created_month")["is_merged"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "merged", "count": "total"})
    .reset_index()
)
monthly_all["acceptance_rate"] = monthly_all["merged"] / monthly_all["total"] * 100

out("Monthly acceptance rate (all 4 agents, own data):")
out(f"{'Month':<12} {'Merged':>8} {'Total':>8} {'Rate':>8}")
out("-" * 40)
for _, row in monthly_all.iterrows():
    out(f"{str(row['created_month']):<12} {int(row['merged']):>8,} {int(row['total']):>8,} {row['acceptance_rate']:>7.1f}%")

In [ ]:
# Monthly acceptance rate — overall line chart
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_all["created_month"].astype(str), monthly_all["acceptance_rate"],
        "o-", color="#2ca02c", linewidth=2, markersize=7, label="All 4 Agents")
ax.set_xlabel("Month")
ax.set_ylabel("Acceptance Rate (%)")
ax.set_title("Monthly Acceptance Rate — Own Data Fix PRs (AIDev Months)")
ax.set_ylim(0, 105)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
fig.savefig(FIG_DIR / "monthly_acceptance_rate.png", dpi=150, bbox_inches="tight")
plt.show()
out(f"\n  -> Figure saved: {FIG_DIR / 'monthly_acceptance_rate.png'}")

In [ ]:
# Monthly acceptance rate — per agent
per_agent_monthly = (
    fix_prs.groupby(["created_month", "agent"])["is_merged"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "merged", "count": "total"})
    .reset_index()
)
per_agent_monthly["acceptance_rate"] = per_agent_monthly["merged"] / per_agent_monthly["total"] * 100

agent_names = sorted(per_agent_monthly["agent"].unique())
palette = sns.color_palette("tab10", len(agent_names))

fig, ax = plt.subplots(figsize=(13, 6))
for agent_tool, color in zip(agent_names, palette):
    subset = per_agent_monthly[per_agent_monthly["agent"] == agent_tool].sort_values("created_month")
    ax.plot(
        subset["created_month"].astype(str),
        subset["acceptance_rate"],
        "o-", label=agent_tool, color=color, linewidth=2, markersize=5,
    )

ax.set_xlabel("Month")
ax.set_ylabel("Acceptance Rate (%)")
ax.set_title("Monthly Acceptance Rate per Agent — Own Data (AIDev Months)")
ax.set_ylim(0, 105)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.legend(title="Agent", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
fig.savefig(FIG_DIR / "monthly_acceptance_rate_per_agent.png", dpi=150, bbox_inches="tight")
plt.show()
out(f"  -> Figure saved: {FIG_DIR / 'monthly_acceptance_rate_per_agent.png'}")

## Monthly Volume

In [ ]:
section("Monthly Fix PR Volume")

monthly_vol = fix_prs.groupby(["created_month", "agent"]).size().unstack(fill_value=0)
monthly_vol["Total"] = monthly_vol.sum(axis=1)

out(f"{'Month':<12}" + "".join(f"{a:>14}" for a in monthly_vol.columns))
out("-" * (12 + 14 * len(monthly_vol.columns)))
for period in monthly_vol.index:
    row = monthly_vol.loc[period]
    out(f"{str(period):<12}" + "".join(f"{int(row[c]):>14,}" for c in monthly_vol.columns))

In [ ]:
# Monthly volume stacked bar chart
fig, ax = plt.subplots(figsize=(12, 5))
agent_cols = [c for c in monthly_vol.columns if c != "Total"]
palette_vol = sns.color_palette("tab10", len(agent_cols))
months_str = [str(p) for p in monthly_vol.index]

bottom = np.zeros(len(months_str))
for agent_name, color in zip(agent_cols, palette_vol):
    vals = monthly_vol[agent_name].values.astype(float)
    ax.bar(months_str, vals, bottom=bottom, label=agent_name, color=color)
    bottom += vals

ax.set_xlabel("Month")
ax.set_ylabel("Number of Fix PRs")
ax.set_title("Monthly Fix PR Volume by Agent — Own Data (AIDev Months)")
ax.legend(title="Agent", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
fig.savefig(FIG_DIR / "monthly_volume_by_agent.png", dpi=150, bbox_inches="tight")
plt.show()
out(f"\n  -> Figure saved: {FIG_DIR / 'monthly_volume_by_agent.png'}")

## Top Repositories

In [ ]:
section("Top 15 Repositories by Fix PRs")

top_repos = fix_prs["repo_name"].value_counts().head(15)
out(f"{'Repo':<55} {'Total':>7} {'Merged':>8} {'Rate':>8}")
out("-" * 80)
for repo, cnt in top_repos.items():
    repo_fix = fix_prs[fix_prs["repo_name"] == repo]
    m = repo_fix["is_merged"].sum()
    r = m / cnt * 100 if cnt else 0
    out(f"  {repo:53s} {cnt:>7,} {m:>8,} {r:>7.1f}%")

## Side-by-Side Comparison with AIDev

Load the AIDev acceptance rates for the same 4 agents (from the AIDev notebook) and compare.

In [ ]:
import re as _re

FIX_PATTERN = _re.compile(
    r"(^fix(\([^)]*\))?!?[\s:!/])"       # conventional: fix:, fix(scope):, fix!:
    r"|(\bfix(es|ed|ing)?\b)"             # natural language: fix, fixes, fixed, fixing
    r"|(\bbug\s*fix\b)"                   # "bugfix" or "bug fix"
    r"|(\bhotfix\b)"                      # hotfix
    r"|(\bbug\b)"                         # bug
    r"|(\bpatch\b)"                       # patch
    r"|(\bresolv(e[ds]?|ing)\b)"          # resolve, resolved, resolves, resolving
    r"|(\berror\b)"                       # error
    r"|(\bcrash(es|ed|ing)?\b)"           # crash, crashes, crashed, crashing
    r"|(\bdefect\b)"                      # defect
    r"|(\bregression\b)"                  # regression
    r"|(\bbroken\b)",                     # broken
    flags=_re.IGNORECASE,
)

def classify_title(title):
    if not title or not isinstance(title, str):
        return "other"
    return "fix" if FIX_PATTERN.search(title) else "other"


section("Side-by-Side Comparison: Own Data vs AIDev")

# Load AIDev fix PRs for the same 4 agents
print("Loading AIDev data for comparison ...")
aidev_all = pd.read_parquet(
    f"{AIDEV_BASE}/all_pull_request.parquet",
    filters=[("agent", "in", AGENTS_STUDIED)],
)
aidev_all["type"] = aidev_all["title"].apply(classify_title)
aidev_fix = aidev_all[(aidev_all["type"] == "fix") & (aidev_all["state"] == "closed")].copy()
aidev_fix["is_merged"] = aidev_fix["merged_at"].notna()

# Build comparison table
out(f"{'Agent':<20} {'AIDev #':>8} {'AIDev %Acc':>12} {'Own #':>8} {'Own %Acc':>12} {'Diff':>8}")
out("-" * 70)

comparison_rows = []
for agent_name in sorted(AGENTS_STUDIED):
    # AIDev
    a_grp = aidev_fix[aidev_fix["agent"] == agent_name]
    a_m, a_t, a_r = merge_rate(a_grp)
    # Own data
    o_grp = fix_prs[fix_prs["agent"] == agent_name]
    o_m, o_t, o_r = merge_rate(o_grp)
    diff = o_r - a_r
    out(f"  {agent_name:18s} {a_t:>8,} {a_r:>11.1f}% {o_t:>8,} {o_r:>11.1f}% {diff:>+7.1f}pp")
    comparison_rows.append({"Agent": agent_name, "AIDev_Total": a_t, "AIDev_Acc%": a_r,
                            "Own_Total": o_t, "Own_Acc%": o_r, "Diff_pp": diff})

# Overall
a_m_all, a_t_all, a_r_all = merge_rate(aidev_fix)
o_m_all, o_t_all, o_r_all = merge_rate(fix_prs)
diff_all = o_r_all - a_r_all
out(f"  {'ALL 4 AGENTS':18s} {a_t_all:>8,} {a_r_all:>11.1f}% {o_t_all:>8,} {o_r_all:>11.1f}% {diff_all:>+7.1f}pp")

comparison_df = pd.DataFrame(comparison_rows)
print("\n")
display(comparison_df)

In [ ]:
# Side-by-side grouped bar chart
agents_cmp = sorted(AGENTS_STUDIED) + ["ALL 4"]
aidev_rates = [comparison_df.loc[comparison_df["Agent"] == a, "AIDev_Acc%"].values[0] for a in sorted(AGENTS_STUDIED)] + [a_r_all]
own_rates   = [comparison_df.loc[comparison_df["Agent"] == a, "Own_Acc%"].values[0]   for a in sorted(AGENTS_STUDIED)] + [o_r_all]

x = np.arange(len(agents_cmp))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 6))
bars1 = ax.bar(x - width / 2, aidev_rates, width, label="AIDev", color="#1f77b4")
bars2 = ax.bar(x + width / 2, own_rates,   width, label="Own Data", color="#ff7f0e")

for bar, val in zip(bars1, aidev_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val:.1f}%", ha="center", fontsize=9, fontweight="bold")
for bar, val in zip(bars2, own_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f"{val:.1f}%", ha="center", fontsize=9, fontweight="bold")

ax.set_ylabel("Acceptance Rate (%)")
ax.set_title("Fix PR Acceptance Rate — AIDev vs Own Data (Same Months)")
ax.set_xticks(x)
ax.set_xticklabels(agents_cmp)
ax.set_ylim(0, 110)
ax.legend(fontsize=11)
ax.grid(axis="y", linestyle="--", alpha=0.4)
fig.tight_layout()
fig.savefig(FIG_DIR / "aidev_vs_own_acceptance.png", dpi=150, bbox_inches="tight")
plt.show()
out(f"\n  -> Figure saved: {FIG_DIR / 'aidev_vs_own_acceptance.png'}")

## PR ID Overlap Analysis

How many PRs are shared between AIDev and our re-collected data?

In [ ]:
section("PR ID Overlap")

aidev_ids = set(aidev_fix["id"])
own_ids = set(fix_prs["id"])

overlap = aidev_ids & own_ids
only_aidev = aidev_ids - own_ids
only_own = own_ids - aidev_ids

out(f"AIDev fix PR IDs   : {len(aidev_ids):,}")
out(f"Own data fix PR IDs: {len(own_ids):,}")
out(f"Overlap            : {len(overlap):,}")
out(f"Only in AIDev      : {len(only_aidev):,}")
out(f"Only in Own data   : {len(only_own):,}")

if len(overlap) > 0:
    # Compare acceptance for overlapping PRs
    aidev_overlap = aidev_fix[aidev_fix["id"].isin(overlap)]
    own_overlap = fix_prs[fix_prs["id"].isin(overlap)]
    a_m, a_t, a_r = merge_rate(aidev_overlap)
    o_m, o_t, o_r = merge_rate(own_overlap)
    out(f"\nOverlapping PRs acceptance:")
    out(f"  AIDev:    {a_m:,} / {a_t:,} = {a_r:.1f}%")
    out(f"  Own data: {o_m:,} / {o_t:,} = {o_r:.1f}%")

## Save Report

In [ ]:
REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")
print(f"\nReport saved to {REPORT_PATH}")
print(f"Figures saved to {FIG_DIR}/")
print("Done!")